In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize, LabelEncoder
from scipy import stats
import numpy as np
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ
#BATTERY FILURE  НЕ ТРОГАТЬ

df = pd.read_csv('C:/Users/rolek/Desktop/vehicles.csv.').drop_duplicates()
df = df.drop(columns=['vehicle_id', 'battery_serial'], errors='ignore')

textcolonki = [
    'vehicle_type',
    'battery_manufacturer',
    'battery_chemistry',
    'drive_type',
    'fleet_or_private',
    'terrain_type',
]

df['vehicle_brand'] = df['vehicle_brand'].fillna(df.groupby('vehicle_model')['vehicle_brand'].transform('first'))
df['vehicle_brand'] = df['vehicle_brand'].fillna(df['vehicle_brand'].mode()[0])

df['vehicle_model'] = df['vehicle_model'].fillna(df.groupby('vehicle_brand')['vehicle_model'].transform( lambda x: x.mode()[0] if not x.mode().empty else np.nan))
df['vehicle_model'] = df['vehicle_model'].fillna(df['vehicle_model'].mode()[0])

for col in textcolonki:
  df[col] = (
      df[col]
      .astype(str)
      .str.strip()
      .str.lower()
      .replace({'nan': np.nan, 'none': np.nan})
  )
  mode_model = df.groupby('vehicle_model')[col].transform(lambda x: x.mode()[0] if not x.mode().empty else np.nan)
  df[col] = df[col].fillna(mode_model).fillna(df[col].mode()[0])

# Год и Пробег медиана по связке (бренд + модель)
df['manufacturing_year'] = df['manufacturing_year'].fillna(df.groupby(['vehicle_brand', 'vehicle_model'])['manufacturing_year'].transform('median'))
df['manufacturing_year'] = df['manufacturing_year'].fillna(df['manufacturing_year'].median())

df['odometer_km'] = df['odometer_km'].fillna(df.groupby(['manufacturing_year', 'vehicle_model'])['odometer_km'].transform('median'))
df['odometer_km'] = df['odometer_km'].fillna(df.groupby('manufacturing_year')['odometer_km'].transform('median'))
df['odometer_km'] = df['odometer_km'].fillna(df['odometer_km'].median())

#  Все остальное числовое (емкость напряжения индексы износа aging_score и тд)
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Все остальные текстовые колонки
obj_cols = df.select_dtypes(include=['object', 'category']).columns
for col in obj_cols:
  df[col] = df[col].fillna(df[col].mode()[0])

df = df.dropna()


outlier_cols = [
    'odometer_km',            
    'battery_capacity_kwh',    
    'cell_voltage_avg',        
    'pack_voltage',            
    'cell_temperature_max',    
    'internal_resistance',    
    'average_speed',          
    'daily_distance'            
]
mask = pd.Series(True, index=df.index)
for col in outlier_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        mask &= (df[col] >= lower_bound) & (df[col] <= upper_bound)
df = df[mask]

df.to_csv('C:/Users/rolek/Desktop/vehicles_clean_v.csv', index=False)

#Нормализация всех числовых колонок 
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    norm = np.linalg.norm(df[col])
    if norm != 0:
        df[col] = df[col] / norm

# Кодирование категориальных признаков
label_cols = ['vehicle_brand', 'vehicle_model', 'battery_manufacturer']
le = LabelEncoder()
for col in label_cols:
    df[col] = le.fit_transform(df[col].astype(str))

df = pd.get_dummies(df, columns=textcolonki, drop_first=True, dtype=int)

df.to_csv('C:/Users/rolek/Desktop/vehicles_clean.csv', index=False)
print(f"размер: {df.shape}")

KeyboardInterrupt: 

In [ ]:
!pip install seaborn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

input_path = "vehicles_clean_vizualizacija.csv"
df_viz = pd.read_csv(input_path)

numeric = [
    c
    for c in [
        "odometer_km",
        "manufacturing_year",
        "battery_capacity_kwh",
        "cell_voltage_avg",
        "pack_voltage",
        "cell_temperature_max",
        "internal_resistance",
        "average_speed",
        "daily_distance",
        "price",
    ]
    if c in df_viz.columns
]

sns.pairplot(df_viz[numeric])
plt.suptitle("Матрица диаграмм рассеяния", y=1.02)
plt.show()

for feature in numeric:
    plt.figure(figsize=(8, 3))
    sns.boxplot(x=df_viz[feature])
    plt.title(f"Boxplot: {feature}")
    plt.tight_layout()
    plt.show()

plt.figure(figsize=(10, 7))
corr = df_viz[numeric].corr(method="spearman")
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Корреляционная матрица Спирмена")
plt.tight_layout()
plt.show()

for feature in numeric:
    plt.figure(figsize=(8, 5))
    sns.histplot(data=df_viz, x=feature, bins=30, kde=True) #Кде это  плотность арспредленияя непрерываня линия
    plt.title(f"Гистограмма: {feature}")
    plt.tight_layout()
    plt.show()

if "manufacturing_year" in df_viz.columns and "odometer_km" in df_viz.columns:
    plt.figure(figsize=(10, 7))
    sns.scatterplot(
        data=df_viz, x="manufacturing_year", y="odometer_km", alpha=0.5
    )
    plt.title("Зависимость пробега от года выпуска")
    plt.xlabel("Год выпуска")
    plt.ylabel("Пробег (км)")
    plt.tight_layout()
    plt.show()

if "battery_capacity_kwh" in df_viz.columns and "odometer_km" in df_viz.columns:
    plt.figure(figsize=(10, 7))
    sns.scatterplot(
        data=df_viz, x="odometer_km", y="battery_capacity_kwh", alpha=0.5
    )
    plt.title("Зависимость емкости батареи от пробега")
    plt.xlabel("Пробег (км)")
    plt.ylabel("Емкость батареи (кВт·ч)")
    plt.tight_layout()
    plt.show()